# 1단계: Colab 환경 + Node 서버 뼈대 + Cloudflared

아래 셀을 **위에서부터 순서대로** 실행하세요. 마지막 터널 셀은 계속 켜 두면 공개 URL이 유지됩니다.

## 1) Node.js 설치

In [ ]:
# Node.js 18.x 설치 (Colab/Ubuntu)
!curl -fsSL https://deb.nodesource.com/setup_18.x | bash -
!apt-get install -y nodejs
!node -v && !npm -v

## 2) 프로젝트 폴더 생성 (신문 플랫폼 / news-platform)

In [ ]:
# 로컬 d:\3.16\news-platform 과 동일한 구조를 Colab /content/news-platform 에 생성
import os
os.makedirs('/content/news-platform', exist_ok=True)

package_json = '''{
  "name": "chungjeong-news",
  "version": "1.0.0",
  "description": "충정일보 신문 플랫폼 — Colab + Node + Cloudflared + MariaDB",
  "main": "server.js",
  "scripts": {
    "start": "node server.js"
  },
  "dependencies": {
    "express": "^4.18.2",
    "mysql2": "^3.6.5"
  }
}
'''

server_js = '''const express = require(\'express\');
const app = express();
const PORT = 3002;

app.get(\'/\', (req, res) => {
  res.send(\'충정일보 서버가 실행 중입니다.\');
});

app.listen(PORT, () => {
  console.log(\`서버가 포트 ${PORT}에서 작동 중입니다.\`);
});
'''

with open('/content/news-platform/package.json', 'w') as f:
    f.write(package_json)
with open('/content/news-platform/server.js', 'w') as f:
    f.write(server_js)

print('package.json, server.js 생성 완료')

## 3) npm install

In [ ]:
!cd /content/news-platform && npm install

## 4) 서버 백그라운드 실행 (포트 3002)

In [ ]:
# 서버를 백그라운드로 실행 (node server.js)
!cd /content/news-platform && nohup node server.js > server.log 2>&1 &
!sleep 2
!cat /content/news-platform/server.log

## 5) Cloudflared 설치 (Linux/Colab용)

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

## 6) 터널 실행 — 공개 URL 확인

아래 셀을 실행하면 `https://xxx.trycloudflare.com` 주소가 출력됩니다. **이 셀을 끄지 말고 켜 두면** 해당 URL로 외부 접속이 가능합니다.

In [ ]:
# localhost:3002 를 터널링 (이 셀은 계속 실행 상태로 두세요)
!cloudflared tunnel --url http://127.0.0.1:3002